# Poetry Generation - Decoder-Only Transformer
---

In [79]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader,Dataset
from collections import Counter
import random
import re
SEED = 207
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. Data 

In [80]:
df = pd.read_csv("hf://datasets/merve/poetry/poetry.csv")

In [81]:
# df.describe()
# df.info()

In [82]:
raw_txt = df['content'].str.cat()
print(f"Total raw string length: {len(raw_txt)}")

Total raw string length: 567201


In [83]:
print(f"Total number of unique characters: {len(set(raw_txt))}")
print(f" Unique characters and frequencies: {Counter(raw_txt)} ")
raw_txt = raw_txt.replace('\r\n','\n')

for i in  re.finditer(r'\{',raw_txt):
    print(raw_txt[i.start()-5:i.start()+5])


Total number of unique characters: 83
 Unique characters and frequencies: Counter({' ': 96319, 'e': 53496, 't': 33938, 'o': 30858, 'a': 28323, 'h': 27705, 's': 27692, 'n': 26580, 'r': 24976, 'i': 24694, 'l': 19498, 'd': 17426, '\r': 14437, '\n': 14437, 'u': 11880, ',': 10531, 'm': 10286, 'w': 9691, 'y': 9615, 'g': 8852, 'f': 8565, 'c': 7809, 'b': 6152, 'p': 5864, 'v': 4552, '.': 3513, 'k': 3464, 'T': 2642, 'A': 2635, 'I': 2183, 'W': 1531, ';': 1408, "'": 1347, 'S': 1249, 'B': 878, 'O': 836, 'H': 798, '-': 741, ':': 735, 'L': 682, 'C': 678, 'F': 649, 'M': 648, 'P': 555, 'N': 473, '?': 451, 'D': 401, 'x': 367, 'E': 355, '!': 321, 'Y': 317, 'j': 290, 'G': 283, 'q': 261, 'R': 253, 'z': 243, '"': 241, '9': 213, '1': 209, 'U': 184, 'V': 132, '(': 110, ')': 109, 'K': 99, 'J': 97, '3': 55, '8': 52, '2': 50, '5': 42, '6': 42, '4': 35, '/': 28, 'Q': 27, 'X': 20, '&': 17, '0': 13, '{': 12, '}': 12, '7': 12, '[': 10, ']': 10, 'Z': 6, '_': 1}) 
ymen {i}{_
n {i}{_o} 
r Pha{"e}t
ke Ph{oe}b
we Ph{oe}b

In [84]:
SPECIAL_MAP = [('{oe}', 'œ'),('{"e}', 'ë'),('{.}', 'ye'),('{i}{_o}', 'io'),('{_o}', 'o'),('{i}', 'i')]
for old,new in SPECIAL_MAP:
    raw_txt = raw_txt.replace(old,new)

print(f"Final length: {len(raw_txt)}")
print(f"Unique characters: {sorted(set(raw_txt))}")

Final length: 552733
Unique characters: ['\n', ' ', '!', '"', '&', "'", '(', ')', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'ë', 'œ']


## 2. Tokenisation


In [85]:
def tokenize(text):
    tokens, word = [], ''  
    for c in text:
        if c == '\n':
            if word:
                tokens.append(word)
                word = ''  
            tokens.append('<EOL>')
        elif c.isalpha():
            word += c
        elif c.isspace():
            if word:
                tokens.append(word)
                word = ''
        else:
            if word:
                tokens.append(word)
                word = ''  
            tokens.append(c)
    if word:
        tokens.append(word)
    return tokens
all_poems = []
for poem in df['content']:
    poem = poem.replace('\r\n', '\n')
    for old, new in SPECIAL_MAP:
        poem = poem.replace(old, new)
    poem = poem.strip()
    if poem:
        all_poems.append(poem)

vocab = ['<BOS>', '<EOS>', '<PAD>', '<EOL>'] + sorted(set(tokenize('\n'.join(all_poems))))
stoi = {t: i for i, t in enumerate(vocab)}
itos = {i: t for t, i in stoi.items()}
vocab_size = len(vocab)
bos_idx, eos_idx, pad_idx = 0, 1, 2

def encode(text):
    return [bos_idx] + [stoi[t] for t in tokenize(text)] + [eos_idx]

def decode(ids):
    out, prev_word = [], False
    for i in ids:
        if i in (bos_idx, eos_idx, pad_idx):
            continue
        t = itos[i]
        if t == '<EOL>':
            out.append('\n')
            prev_word = False
        elif len(t) == 1 and not t.isalnum():
            out.append(t)
            prev_word = False
        else:
            if prev_word:
                out.append(' ')
            out.append(t)
            prev_word = True
    return ''.join(out)

n_poems = len(all_poems)
train_poems = all_poems[:int(0.8 * n_poems)]
val_poems = all_poems[int(0.8 * n_poems):int(0.9 * n_poems)]
test_poems = all_poems[int(0.9 * n_poems):]

train_encoded = [encode(p) for p in train_poems]
val_encoded = [encode(p) for p in val_poems]
test_encoded = [encode(p) for p in test_poems]

print(f'Vocabulary size: {vocab_size}')
print(f'Sample tokens: {train_encoded[0][:100]}')

Vocabulary size: 13107
Sample tokens: [0, 1430, 11681, 3718, 8830, 8054, 7779, 26, 1753, 11681, 10889, 157, 11984, 26, 1151, 10223, 3184, 12052, 3531, 10, 26, 2555, 12733, 10949, 4299, 12787, 8793, 12, 26, 26, 373, 11753, 10627, 6947, 10, 26, 958, 9450, 8830, 11681, 6101, 10, 26, 201, 8830, 11681, 6091, 7, 10215, 5686, 10, 26, 2555, 11742, 12029, 4537, 11753, 8738, 8635, 12, 26, 26, 972, 11742, 10475, 7466, 26, 836, 6411, 8830, 12124, 12783, 10, 26, 2134, 11681, 5553, 10, 6041, 7, 4915, 7636, 25, 26, 1334, 11681, 8805, 10871, 11328, 12, 26, 26, 1430, 11681, 9499, 7367, 11486, 12721, 10, 26]


## 3. Dataset

In [86]:
l = [len(i) for i in raw_txt.split('\n') if len(i) != 0]
np.percentile(l,99.5)

np.float64(112.0)

In [87]:
SEQ_LEN = 128
BATCH = 64

class PoetryDataset(Dataset):
    def __init__(self, encoded_poems, seq_len=SEQ_LEN, pad_idx=pad_idx):
        self.pad_idx = pad_idx
        self.seq_len = seq_len
        self.data = []
        for poem in encoded_poems:
            if len(poem) <= seq_len:
                padded = poem + [pad_idx] * (seq_len - len(poem))
                self.data.append(padded)
            else:
                for i in range(0, len(poem) - seq_len, seq_len):
                    self.data.append(poem[i:i+seq_len])
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, i):
        x = torch.tensor(self.data[i][:-1], dtype=torch.long)
        y = torch.tensor(self.data[i][1:], dtype=torch.long)
        return x, y

train_loader = DataLoader(PoetryDataset(train_encoded), batch_size=BATCH, shuffle=True)
val_loader = DataLoader(PoetryDataset(val_encoded), batch_size=BATCH, shuffle=False)
test_loader = DataLoader(PoetryDataset(test_encoded), batch_size=BATCH, shuffle=False)

## 4. Baselines

### 4.1 Models

In [88]:
class BigramModel:
    def __init__(self, vocab_size):
        self.vocab_size = vocab_size
        self.counts = {}
    
    def fit(self, tokens):
        for i,j in zip(tokens,tokens[1:]):
            self.counts.setdefault(i, {})[j] = self.counts.get(i ,{}).get(j, 0) + 1
    
    def dist(self,prev):
        if prev not in self.counts:
            return {}
        next_cnts = self.counts[prev]
        next_sum = sum(next_cnts.values())
        probs = {}
        for i in next_cnts:
            probs[i] = next_cnts[i]/next_sum
        return probs 

    def generate(self, start_token, max_len=20):
        out = [start_token]
        current = start_token
        for i in range(max_len-1):
            probs = self.dist(current)
            candidates = list(probs.keys())
            weights = list(probs.values())
            next_tok = np.random.choice(candidates,p=weights)
            if next_tok is None:
                break
            out.append(next_tok)
            current = next_tok
        return out 

    def perplexity(self, tokens):
        log_prb = 0
        n = 0
        for i,j in zip(tokens, tokens[1:]):
            probs = self.dist(i)
            probs = probs.get(j,1e-6)
            log_prb += np.log(probs)
            n += 1 
        return np.exp(-log_prb/n)

bigram_model = BigramModel(vocab_size=vocab_size)    
bigram_model.fit([t for poem in train_encoded for t in poem])
out = bigram_model.generate(random.choice([t for poem in val_encoded for t in poem]))
ppl = bigram_model.perplexity([t for poem in test_encoded for t in poem])
print(f" Bigram Baseline Model")
print('-'*60)
print("Generated text from random token\n")
print(decode(out))   
print()     
print(f"Perplexity score : {ppl}")

 Bigram Baseline Model
------------------------------------------------------------
Generated text from random token

the thing to pine,the nunnery beaches
For lo the thought
Seeing iron,yea,behind

Perplexity score : 1552.8620094680516


In [89]:
D_MODEL = 128
N_HEADS = 4
class MLPBaseline(nn.Module):
    def __init__(self, vocab_size=vocab_size, d_model=D_MODEL, hidden_dim=[128], n_classes=vocab_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        layers, cur_dim = [], d_model
        for h in hidden_dim:
            layers += [nn.Linear(cur_dim, h), nn.ReLU()]
            cur_dim = h
        layers.append(nn.Linear(cur_dim, n_classes))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x):
        return self.mlp(self.embed(x))

class GRUBaseline(nn.Module):
    def __init__(self, vocab_size=vocab_size, d_model=D_MODEL, hidden=128, n_classes=vocab_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.gru = nn.GRU(d_model, hidden, 1, batch_first=True)
        self.head = nn.Linear(hidden, n_classes)
    def forward(self, x):
        out, _ = self.gru(self.embed(x))
        return self.head(out)


### 4.2 Training and Eval utils

In [90]:
def train_epoch(model,loader,optimiser,criterion,device=device):
    model.train()
    tot_loss = 0
    for x,y in loader:
        x = x.to(device)
        y = y.to(device)
        optimiser.zero_grad()
        logits = model(x)
        loss = criterion(logits.view(-1,vocab_size), y.view(-1))
        loss.backward()
        optimiser.step()
        tot_loss += loss.item()
    return tot_loss / len(loader)

def evaluate(model, loader, criterion=nn.CrossEntropyLoss(ignore_index=pad_idx), device=device):
    model.eval()
    tot_loss = 0
    for x,y in loader:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        loss = criterion(logits.view(-1,vocab_size), y.view(-1))
        tot_loss += loss.item()
    avg_loss = tot_loss / len(loader)
    return {'loss': avg_loss, 'ppl': np.exp(avg_loss)}

def train(model, train_loader, val_loader,
          epochs = 30,
          lr = 3e-4,
          device = device,
          optim = torch.optim.Adam,
          crit = nn.CrossEntropyLoss,
          verbose=False,
          probe_seq=None,
          snapshot_every=5,
          **kwargs):
    model = model.to(device)
    optimizer = optim(model.parameters(), lr=lr, **kwargs)
    criterion = crit(ignore_index=pad_idx)
    history = []
    snapshots = []
    best_val = np.inf

    probe_x = None
    if probe_seq is not None:
        probe_x = torch.tensor([probe_seq], dtype=torch.long).to(device)

    for epoch in range(epochs):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        val_res = evaluate(model, val_loader, criterion, device)
        history.append({'epoch':epoch+1, 'train_loss':train_loss, **val_res})

        if val_res['loss'] < best_val:
            best_val  = val_res['loss']
            torch.save(model.state_dict(),f'best{model.__class__.__name__}.pt')

        if probe_x is not None and (epoch)%snapshot_every == 0:
            model.eval()
            with torch.no_grad():
                _, all_weights = model.forward_with_weights(probe_x)
                snapshots.append((epoch +1, [w[0].cpu() for w in all_weights]))
            model.train()
        if (epoch)%5 == 0 and verbose:
            print(f"Epoch {epoch+1} | Train : {train_loss:.2f} | Val : {val_res['loss']:.2f} | PPL : {val_res['ppl']:.2f}")

    return history, snapshots

In [91]:
print("MLP Baseline")
print('-'*60)
mlp_model = MLPBaseline(vocab_size=vocab_size,d_model=256,hidden_dim=[256,128])
mlp_history,_= train(mlp_model, train_loader,val_loader, verbose=True, lr=1e-3)
mlp_model.load_state_dict(torch.load('bestMLPBaseline.pt', weights_only=True))
test_res = evaluate(mlp_model, test_loader, device=device)
print(f'Test loss: {test_res["loss"]} | ppl: {test_res["ppl"]}')

MLP Baseline
------------------------------------------------------------
Epoch 1 | Train : 8.88 | Val : 7.75 | PPL : 2332.70
Epoch 6 | Train : 5.73 | Val : 6.17 | PPL : 476.77
Epoch 11 | Train : 5.28 | Val : 6.13 | PPL : 460.10
Epoch 16 | Train : 4.87 | Val : 6.29 | PPL : 538.55
Epoch 21 | Train : 4.52 | Val : 6.52 | PPL : 680.29
Epoch 26 | Train : 4.26 | Val : 6.74 | PPL : 849.23
Test loss: 6.213197469711304 | ppl: 499.2951828790641


In [111]:
print("GRU Baseline")
print('-'*60)
gru_model = GRUBaseline(vocab_size=vocab_size,d_model=512)
gru_history,_ = train(gru_model, train_loader,val_loader, verbose=True,epochs=100)
gru_model.load_state_dict(torch.load('bestGRUBaseline.pt', weights_only=True))
test_res = evaluate(gru_model, test_loader, device=device)
print(f'Test loss: {test_res["loss"]} | ppl: {test_res["ppl"]}')

GRU Baseline
------------------------------------------------------------
Epoch 1 | Train : 9.38 | Val : 9.23 | PPL : 10178.48
Epoch 6 | Train : 6.58 | Val : 6.68 | PPL : 793.25
Epoch 11 | Train : 6.19 | Val : 6.40 | PPL : 598.97
Epoch 16 | Train : 6.07 | Val : 6.30 | PPL : 547.11
Epoch 21 | Train : 5.96 | Val : 6.22 | PPL : 500.33
Epoch 26 | Train : 5.83 | Val : 6.14 | PPL : 462.06
Epoch 31 | Train : 5.71 | Val : 6.05 | PPL : 426.17
Epoch 36 | Train : 5.60 | Val : 5.98 | PPL : 395.51
Epoch 41 | Train : 5.47 | Val : 5.91 | PPL : 370.26
Epoch 46 | Train : 5.35 | Val : 5.85 | PPL : 347.66
Epoch 51 | Train : 5.21 | Val : 5.80 | PPL : 329.39
Epoch 56 | Train : 5.10 | Val : 5.75 | PPL : 313.86
Epoch 61 | Train : 4.98 | Val : 5.72 | PPL : 303.53
Epoch 66 | Train : 4.84 | Val : 5.69 | PPL : 295.94
Epoch 71 | Train : 4.74 | Val : 5.67 | PPL : 290.50
Epoch 76 | Train : 4.63 | Val : 5.65 | PPL : 285.50
Epoch 81 | Train : 4.51 | Val : 5.65 | PPL : 283.63
Epoch 86 | Train : 4.39 | Val : 5.63 | PPL

### 5. Decoder only


In [ ]:
class MaskedMultiHeadAttention(nn.Module):
    def __init__(self, d_model = D_MODEL, n_heads = N_HEADS):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.dk = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model,bias=False)
        self.W_k = nn.Linear(d_model, d_model,bias=False)
        self.W_v = nn.Linear(d_model, d_model,bias=False)
        self.W_o = nn.Linear(d_model, d_model,bias=False)

    def forward(self, x):
        batch, seq_len,_ = x.shape 
        H = self.n_heads
        dk = self.dk
        Q = self.W_q(x).view(batch,seq_len,H,dk).transpose(1,2)
        K = self.W_k(x).view(batch,seq_len,H,dk).transpose(1,2)
        V = self.W_v(x).view(batch,seq_len,H,dk).transpose(1,2)
        attn_scores = (Q @ K.transpose(-2,-1)) / (dk**0.5)
        mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1).bool()
        attn_scores = attn_scores.masked_fill(mask, float('-inf'))
        weights = torch.softmax(attn_scores, dim=-1)
        out = weights @ V
        out = out.transpose(1,2).contiguous().view(batch,seq_len,-1)
        return self.W_o(out), weights

class TransformerBlock(nn.Module):
        def __init__(self, d_model=D_MODEL, n_heads=N_HEADS, hidden=256,drop=0.1):
             super().__init__()
             self.attn = MaskedMultiHeadAttention(d_model,n_heads)
             self.ln1 = nn.LayerNorm(d_model)
             self.ln2 = nn.LayerNorm(d_model)
             self.ff = nn.Sequential(nn.Linear(d_model,hidden),nn.ReLU(),nn.Linear(hidden,d_model))
             self.drop = nn.Dropout(drop)
        def forward(self, x):
            attn_out, w = self.attn(self.ln1(x))
            x = x+self.drop(attn_out)
            x = x + self.ff(self.ln2(x))
            return x, w

class DecoderOnlyTransformer(nn.Module):
        def __init__(self, d_model = D_MODEL, n_heads = N_HEADS, n_classes=vocab_size,
                     n_layers = 4,
                     hidden = 256,
                     use_pe = True,
                     drop=0.1):
            super().__init__()

            self.embed = nn.Embedding(vocab_size, d_model)
            self.blocks = nn.ModuleList([TransformerBlock(d_model, n_heads, hidden,drop) for _ in range(n_layers)])
            self.use_pe = use_pe
            self.head = nn.Linear(d_model, n_classes)
            if use_pe:
                 self.pe = nn.Embedding(SEQ_LEN,d_model)
        
        def _embed(self, x):
            x = self.embed(x)
            if self.use_pe:
                pos = torch.arange(x.shape[1], device=x.device).unsqueeze(0)
                x = x + self.pe(pos)
            return x

        def forward(self, x):
            x = self._embed(x)
            for block in self.blocks:
                x, _ = block(x)
            return self.head(x)

        def forward_with_weights(self, x):
            x = self._embed(x)
            all_weights = []
            for block in self.blocks:
                x, w = block(x)
                all_weights.append(w)
            return self.head(x), all_weights

## 5. Evaluation

In [143]:
print(" Decoder Transformer ")
print('-'*60)
model = DecoderOnlyTransformer(n_layers=2,d_model=256,n_heads=4,hidden=512,n_classes=vocab_size)
tr_history,snapshots = train(model, train_loader, val_loader, epochs=30, verbose=True,lr=1e-3,weight_decay=0.01)
model.load_state_dict(torch.load('bestDecoderOnlyTransformer.pt', weights_only=True))
test_res = evaluate(model, test_loader, device=device)
print(f"Test loss: {test_res['loss']} | ppl: {test_res['ppl']}")


 Decoder Transformer 
------------------------------------------------------------
Epoch 1 | Train : 7.96 | Val : 6.75 | PPL : 856.86
Epoch 6 | Train : 5.79 | Val : 6.02 | PPL : 411.28
Epoch 11 | Train : 5.70 | Val : 5.97 | PPL : 390.67
Epoch 16 | Train : 5.69 | Val : 5.94 | PPL : 381.12
Epoch 21 | Train : 5.72 | Val : 5.96 | PPL : 386.69
Epoch 26 | Train : 5.73 | Val : 5.95 | PPL : 384.51
Test loss: 6.042335748672485 | ppl: 420.8749457876195


## 6. Decoding

In [ ]:
@torch.no_grad()
def generate(model, prompt, max_new=200, temperature=1.0, top_p=1.0, rep_penalty=1.0):
    model.eval()
    # right shift by 1
    ids = encode(prompt)[:-1]
    ids = torch.tensor([ids], dtype=torch.long, device=device)

    for _ in range(max_new):
        # batch is 1 only and take the last tokens
        logits = model(ids[:, -SEQ_LEN:])[0, -1]

        # apply
        for tok in set(ids[0, -20:].tolist()):
            logits[tok] /= rep_penalty
        
        if temperature == 0:
            next_tok = logits.argmax().item()
        else:
            logits /= temperature
            if top_p < 1.0:
                sorted_logits, sorted_idx = torch.sort(logits, descending=True)
                probs = torch.nn.functional.softmax(sorted_logits, dim=-1)
                remove = probs.cumsum(0) - probs > top_p
                sorted_logits[remove] = float('-inf')
                logits[sorted_idx] = sorted_logits
            next_tok = torch.multinomial(torch.nn.functional.softmax(logits, dim=-1), 1).item()

        if next_tok == eos_idx:
            break
        ids = torch.cat([ids, torch.tensor([[next_tok]], device=device)], dim=1)

    return decode(ids[0].tolist())

In [136]:
prompt = 'And the'
print('-'*60)
print('Greedy')
print(generate(model, prompt, temperature=0))
print('Temperature 0.7')
print('-'*60)
print(generate(model, prompt, temperature=0.7))
print('-'*60)
print('Top-p 0.8 + Temperature 0.8')
print('-'*60)
print(generate(model, prompt, temperature=0.8, top_p=0.8))
print('-'*60)
print('Top-p 0.9 + rep penalty 1.3 + Temperature 0.8')
print('-'*60)
print(generate(model, prompt, temperature=0.8, top_p=0.9, rep_penalty=1.3))


------------------------------------------------------------
Greedy
And the wanderer the Image of the world,
And
The nombers.


And the riper
And the world,
The primitias the world
And the world,



And let

And the studious
And the curbe the avenue

And the sun the bush,

And the woods the roamings,



And the eagle



And the prouder the world
And the vast the price the world,
And the spheres

And lyftes

And the world
And the Southerne
And
And the river




And the edge













































































Temperature 0.7
------------------------------------------------------------
And the wanderer every with water of earthy all the basest strawberries tree
And and weeds the commanding

And we the same,


But

Brings Thames thou of one as the basest he the outward,

My them I if went I have we me

Of the way
The rich in the angling,
For at great
Have

I am the field
And which blaze
And Primroses foison had
So a shade,all the sense.
Till
I know

In [115]:
print('Top-p 0.9 + rep penalty 1.3 (GRU)')
print('-'*60)
print(generate(gru_model, prompt, temperature=0.8, top_p=0.9, rep_penalty=1.3))

Top-p 0.9 + rep penalty 1.3 (GRU)
------------------------------------------------------------
And the woods within my heart,
Her death for thy powr
Of Olive summers night before,
Love,they all my show,and I saw the green heart;
Like fair love-wind my own.
Whilome on those tomb by permission of shade,
And branch's time sweet the towny out with the rest in payne;
It is not a woman,and some!
To put me to no more.
This light as she in such a flower.
Think the flood with delight:
The Her little shall obey;
As you have I must are force and dust,from that now is the world,or my love
And all these's mysteries to in one well.
That the blackbird is indeed,nor by me;
On that they!I know not so?
The rocks-tree out of the morning to fill,
And with the curse of wemens passion;
Let our cooles and majesty.
Like a far-born Eros,
